# Milestone 3 — Preprocessing and frozen partitions

Six original inputs only. Split 60% training / 20% validation / 20% test, stratified on the binary target with seed 42. Validation selects models, thresholds, and calibration; test data is reserved for a single final evaluation. Full-data exploratory exposure is disclosed in the EDA report. A separate grouped development-set check will assess row-order sensitivity without making claims about true chronology.

In [1]:
from pathlib import Path
import json
import hashlib
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/raw/ai4i2020.csv").exists())
source = nbformat.read(PROJECT_ROOT / "notebooks/01_dataset_audit_and_eda.ipynb", as_version=4)
for cell in source.cells:
    if cell.cell_type == "code" and "definitions" in cell.metadata.get("tags", []):
        exec(compile(cell.source, "audit_definitions", "exec"), globals())
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
MODEL_DIR = PROJECT_ROOT / "models"
METRICS_DIR = REPORT_DIR / "metrics"
SPLIT_PATH = PROJECT_ROOT / "data/processed/split_manifest.json"

In [2]:
def make_preprocessor():
    return ColumnTransformer([
        ("numeric", StandardScaler(), list(NUMERIC_COLUMNS)),
        ("type", OneHotEncoder(categories=[["L", "M", "H"]], handle_unknown="error", sparse_output=False), ["Type"]),
    ], remainder="drop", verbose_feature_names_out=False)

def load_partitions():
    records = load_dataset()
    manifest = json.loads(SPLIT_PATH.read_text())
    current_hash = hashlib.sha256(RAW_DATA_PATH.read_bytes()).hexdigest()
    if manifest["source_sha256"] != current_hash:
        raise ValueError("Dataset differs from the frozen split manifest")
    return records, manifest

def get_partition(records, manifest, name):
    part = records.iloc[manifest[name]]
    return part[list(FEATURE_COLUMNS)].copy(), part[TARGET_COLUMN].copy()

In [3]:
records = load_dataset()
indices = np.arange(len(records))
train_indices, remaining = train_test_split(indices, test_size=0.4, stratify=records[TARGET_COLUMN], random_state=RANDOM_STATE)
validation_indices, test_indices = train_test_split(remaining, test_size=0.5, stratify=records.iloc[remaining][TARGET_COLUMN], random_state=RANDOM_STATE)
manifest = {
    "seed": RANDOM_STATE,
    "source_sha256": hashlib.sha256(RAW_DATA_PATH.read_bytes()).hexdigest(),
    "train": sorted(train_indices.tolist()),
    "validation": sorted(validation_indices.tolist()),
    "test": sorted(test_indices.tolist()),
    "grouping_protocol": "CSV position // 1000; grouped cross-validation on training rows only",
}
SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
if SPLIT_PATH.exists():
    assert json.loads(SPLIT_PATH.read_text()) == manifest, "Refusing to change existing partitions"
else:
    SPLIT_PATH.write_text(json.dumps(manifest, indent=2) + "\n")
MODEL_DIR.mkdir(exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
support = []
for name in ("train", "validation", "test"):
    part = records.iloc[manifest[name]]
    support.append({"partition": name, "records": len(part), "failures": int(part[TARGET_COLUMN].sum()), **part[list(FAILURE_COLUMNS)].sum().to_dict()})
display(pd.DataFrame(support))
(METRICS_DIR / "split_support.json").write_text(json.dumps(support, indent=2) + "\n")

,partition,records,failures,TWF,HDF,PWF,OSF,RNF
0,train,6000,203,27,67,64,57,11
1,validation,2000,68,12,22,16,21,3
2,test,2000,68,7,26,15,20,5


455

## Fit on training rows only

Explicit categories give a stable encoding; unseen categories raise an error. Scaling is fitted only on training readings, and the column transformer drops all unlisted columns.

In [4]:
X_train, y_train = get_partition(records, manifest, "train")
X_validation, y_validation = get_partition(records, manifest, "validation")
preprocessor = make_preprocessor()
transformed = preprocessor.fit_transform(X_train)
validation_transformed = preprocessor.transform(X_validation)
assert transformed.shape == (6000, 8)
assert validation_transformed.shape == (2000, 8)
np.testing.assert_allclose(preprocessor.named_transformers_["numeric"].mean_, X_train[list(NUMERIC_COLUMNS)].mean())
assert set(preprocessor.get_feature_names_out()) == set(NUMERIC_COLUMNS) | {"Type_L", "Type_M", "Type_H"}
assert not set(manifest["train"]) & set(manifest["validation"])
assert not set(manifest["train"]) & set(manifest["test"])
assert not set(manifest["validation"]) & set(manifest["test"])
assert sorted(manifest["train"] + manifest["validation"] + manifest["test"]) == list(range(len(records)))
invalid = X_validation.iloc[[0]].copy()
invalid["Type"] = "X"
try:
    preprocessor.transform(invalid)
except ValueError:
    pass
else:
    raise AssertionError("Unsupported category accepted")
print("Preprocessing checks passed: disjoint splits, training-only scaling, explicit inputs, unsupported category rejection.")
display(pd.DataFrame(transformed[:5], columns=preprocessor.get_feature_names_out()))

Preprocessing checks passed: disjoint splits, training-only scaling, explicit inputs, unsupported category rejection.


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Type_L,Type_M,Type_H
0,-0.957898,-0.954692,0.067199,0.275630,-1.691034,0.0,1.0,0.0
1,-0.957898,-1.022286,-0.226405,0.933333,-1.612376,1.0,0.0,0.0
2,-0.908002,-0.954692,-0.586486,-0.053221,-1.580913,1.0,0.0,0.0
3,-0.858106,-0.887098,0.709805,-1.139426,-1.407865,0.0,1.0,0.0
4,-0.808210,-0.751909,1.346871,-1.607790,-1.313476,0.0,0.0,1.0


No model is trained in this notebook. The test partition is inspected only for support, not performance. Subsequent notebooks load the frozen manifest and construct complete preprocessing/model pipelines.